# ExperimentaÃƒÂ§ÃƒÂ£o

Este notebook orquestra a Fase 1 e 2 da etapa de experimentaÃƒÂ§ÃƒÂ£o, seguindo o protocolo descrito em `docs/experimentation.md`.

## Objetivos

## Setups e Imports

In [1]:
import sys
from pathlib import Path

# Garante que a raiz do projeto esta no sys.path
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Manipulacao de Dados
import pandas as pd
import numpy as np

# Visualizacao de Dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pre-processamento e Modelagem
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
)
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Metricas de Avaliacao
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    make_scorer,
)

# Modelos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from src.utils.exp import (
    MLPClassifierWrapper,
    build_k_grid,
    evaluate_round3_model_strategies,
    extract_selected_feature_names,
    format_selected_features_log,
    get_processed_feature_names,
    summarize_grid_search_results,
)

# importando os transformers customizados
from src.features.geo_transformer import GeoTransformer
from src.features.feature_engineer_transformer import FeatureEngineerTransformer

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# MLflow
# ATENCAO: o servidor do MLflow precisa estar rodando antes de executar este notebook.
# Em um terminal separado, execute:
#   mlflow server --host 127.0.0.1 --port 5000
# Sem isso, as celulas de tracking falharao com ConnectionRefusedError.
import hashlib
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")

## Processamento dos Dados

In [2]:
# dataload
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [3]:
# InformaÃƒÂ§ÃƒÂµes gerais sobre o dataset
print("=== INFORMAÃƒâ€¡Ãƒâ€¢ES GERAIS DO DATASET ===\n")
print(df.info())

# Shape
print("\n=== SHAPE DO DATASET ===")
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

# Colunas
print("\n=== COLUNAS DO DATASET ===")
print(df.columns.tolist())

=== INFORMAÃƒâ€¡Ãƒâ€¢ES GERAIS DO DATASET ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 n

In [4]:
# TransformaÃƒÂ§ÃƒÂ£o da coluna 'Total Charges' para numÃƒÂ©rica, tratando erros e preenchendo valores ausentes com 0.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

In [5]:
# dropando colunas irrelevantes para a modelagem
drop_cols = [
    'Country',
    'State',
    'Lat Long',
    'Churn Label',
    'Churn Reason',
    'Count'
]


df.drop(columns=drop_cols, inplace=True)

In [6]:
target = "Churn Value"
meta_cols = ["CLTV", "CustomerID"]

feature_cols = [
    col for col in df.columns
    if col not in [target] + meta_cols
]

X = df[feature_cols]
y = df[target]

metadata = df[meta_cols]


## Splits e ValidaÃƒÂ§ÃƒÂ£o

In [7]:
# Protocolo de validaÃƒÂ§ÃƒÂ£o cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 
X_train_val, X_test, y_train_val, y_test, metadata_train_val, metadata_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Checando os Splits
print("=== SPLITS ===")
print(f"Treino/ValidaÃƒÂ§ÃƒÂ£o: {X_train_val.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

# Checando a distribuiÃƒÂ§ÃƒÂ£o da variÃƒÂ¡vel alvo nos splits
print("\n=== DISTRIBUIÃƒâ€¡ÃƒÆ’O DA VARIÃƒÂVEL ALVO NOS SPLITS ===")
print("Treino/ValidaÃƒÂ§ÃƒÂ£o:")
print(y_train_val.value_counts(normalize=True))
print("\nTeste:")
print(y_test.value_counts(normalize=True))

# Checando o formato dos dados
print("\n=== FORMATO DOS DADOS ===")
print(f"X_train_val: {X_train_val.shape}")
print(f"y_train_val: {y_train_val.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")


=== SPLITS ===
Treino/ValidaÃƒÂ§ÃƒÂ£o: 4930 amostras
Teste: 2113 amostras

=== DISTRIBUIÃƒâ€¡ÃƒÆ’O DA VARIÃƒÂVEL ALVO NOS SPLITS ===
Treino/ValidaÃƒÂ§ÃƒÂ£o:
Churn Value
0    0.734686
1    0.265314
Name: proportion, dtype: float64

Teste:
Churn Value
0    0.734501
1    0.265499
Name: proportion, dtype: float64

=== FORMATO DOS DADOS ===
X_train_val: (4930, 24)
y_train_val: (4930,)
X_test: (2113, 24)
y_test: (2113,)


## Baseline Inicial

- Treina Dummy e RegressÃƒÂ£o LogÃƒÂ­stica e compara com a MLP e outros modelos de ÃƒÂ¡rvores.

In [8]:
# Definindo a etapa de prÃƒÂ©-processamento para variÃƒÂ¡veis categÃƒÂ³ricas com OHE e numÃƒÂ©ricas com passthrough
ohe = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, make_column_selector(dtype_include=["object", "category"])),
        ("num", "passthrough", make_column_selector(dtype_exclude=["object", "category"])),
    ],
    remainder="drop",
)

# DicionÃƒÂ¡rio de Pipelines para cada modelo
baseline_params = dict(
    drop_churn_score=True,
    add_engagement_score=False,
    add_tenure_group=False,
    add_tenure_log=False,
    add_contract_ordinal=False,
    add_family_stability=False,
    add_fiber_no_support=False,
    add_support_gap_count=False,
    add_payment_automatic_flag=False,
    add_electronic_check_flag=False,
    add_paperless_echeck_flag=False,
    add_price_pressure_ratio=False,
)

models = {
    "Dummy": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "DecisionTree": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced",
        )),
    ]),
    "RandomForest": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            class_weight="balanced_subsample",
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**baseline_params)),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}


In [9]:
# definindo o scoring para avaliaÃƒÂ§ÃƒÂ£o dos modelos
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

In [10]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{m: cv_res[f"test_{m}"] for m in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{m}_mean": cv_res[f"test_{m}"].mean() for m in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÃƒâ€°DIOS DA VALIDAÃƒâ€¡ÃƒÆ’O CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: Dummy ===
=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: DecisionTree ===
=== AVALIANDO MODELO: RandomForest ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÃƒâ€°DIOS DA VALIDAÃƒâ€¡ÃƒÆ’O CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6780,0.8575,0.8142,0.5314,0.6430,0.0240,0.0112
1,MLP,0.6780,0.8565,0.8111,0.5152,0.6301,1.1009,0.0141
2,XGBoost,0.6492,0.8438,0.6743,0.5685,0.6167,0.0600,0.0156
3,RandomForest,0.6323,0.8393,0.5091,0.6467,0.5692,0.1692,0.0638
4,DecisionTree,0.3927,0.6684,0.5099,0.5147,0.5120,0.0229,0.0115
5,Dummy,0.2653,0.5000,0.0000,0.0000,0.0000,0.0096,0.0105


### Validando o Wrapper

- AplicaÃƒÂ§ÃƒÂ£o da MLP fora do pipeline com o ciclo manual de validaÃƒÂ§ÃƒÂ£o para validar o resultado do wrapper.

In [11]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)

from src.models.mlp import MLP, evaluate, train_with_early_stopping

# ---------- Config ----------
MLP_EPOCHS = 80
MLP_BATCH_SIZE = 64
MLP_LR = 1e-3
MLP_WD = 1e-5
MLP_HIDDEN_DIM = 64
MLP_THRESHOLD = 0.5

ES_PATIENCE = 8
ES_VAL_SIZE = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _rows(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]


def _to_dense_float32(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x, dtype=np.float32)


baseline_fe = FeatureEngineerTransformer(**baseline_params)
baseline_geo = GeoTransformer(strategy="drop")

mlp_manual_folds = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_val, y_train_val), start=1):
    fit_start = time.perf_counter()

    X_tr_raw = _rows(X_train_val, tr_idx)
    X_va_raw = _rows(X_train_val, va_idx)
    y_tr = np.asarray(_rows(y_train_val, tr_idx), dtype=np.float32)
    y_va = np.asarray(_rows(y_train_val, va_idx), dtype=np.float32)

    X_tr_base = baseline_fe.fit_transform(X_tr_raw, y_tr)
    X_tr_base = baseline_geo.fit_transform(X_tr_base, y_tr)
    X_va_base = baseline_fe.transform(X_va_raw)
    X_va_base = baseline_geo.transform(X_va_base)

    prep_fold = clone(preprocessor)
    X_tr_enc = prep_fold.fit_transform(X_tr_base, y_tr)
    X_va_enc = prep_fold.transform(X_va_base)

    idx_all = np.arange(len(y_tr))
    idx_tr, idx_es = train_test_split(
        idx_all,
        test_size=ES_VAL_SIZE,
        stratify=y_tr,
        random_state=42 + fold,
    )

    X_tr_fit = X_tr_enc[idx_tr]
    y_tr_fit = y_tr[idx_tr]
    X_tr_es = X_tr_enc[idx_es]
    y_tr_es = y_tr[idx_es]

    scaler = StandardScaler(with_mean=False)
    X_tr_fit = scaler.fit_transform(X_tr_fit)
    X_tr_es = scaler.transform(X_tr_es)
    X_va_sc = scaler.transform(X_va_enc)

    X_tr_fit = _to_dense_float32(X_tr_fit)
    X_tr_es = _to_dense_float32(X_tr_es)
    X_va_sc = _to_dense_float32(X_va_sc)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_fit, dtype=torch.float32),
            torch.tensor(y_tr_fit, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
    )
    es_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_es, dtype=torch.float32),
            torch.tensor(y_tr_es, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=False,
    )

    model = MLP(
        input_dim=X_tr_fit.shape[1],
        hidden_dim=MLP_HIDDEN_DIM,
        output_dim=1,
    ).to(DEVICE)

    pos = float((y_tr_fit == 1).sum())
    neg = float((y_tr_fit == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=MLP_WD)

    epochs_trained = train_with_early_stopping(
        model,
        train_loader,
        es_loader,
        optimizer,
        criterion,
        device=DEVICE,
        max_epochs=MLP_EPOCHS,
        patience=ES_PATIENCE,
        threshold=MLP_THRESHOLD,
    )
    best_es_loss, _ = evaluate(
        model,
        es_loader,
        criterion,
        device=DEVICE,
        threshold=MLP_THRESHOLD,
    )
    fit_time_s = time.perf_counter() - fit_start

    score_start = time.perf_counter()
    X_va_t = torch.tensor(X_va_sc, dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_va_t).squeeze(1)
        prob = torch.sigmoid(logits).cpu().numpy()

    pred = (prob >= MLP_THRESHOLD).astype(int)

    mlp_manual_folds.append(
        {
            "fold": fold,
            "epochs_trained": epochs_trained,
            "best_es_loss": best_es_loss,
            "pr_auc": average_precision_score(y_va, prob),
            "roc_auc": roc_auc_score(y_va, prob),
            "recall": recall_score(y_va, pred, zero_division=0),
            "precision": precision_score(y_va, pred, zero_division=0),
            "f1": f1_score(y_va, pred, zero_division=0),
            "fit_time_s": fit_time_s,
            "score_time_s": time.perf_counter() - score_start,
        }
    )

# resultados por fold
mlp_fold_results = pd.DataFrame(mlp_manual_folds)

# mÃƒÂ©dias para comparar com a versÃƒÂ£o fora do pipeline
mlp_cv_summary = pd.DataFrame([
    {
        "model": "MLP_manual",
        "pr_auc_mean": mlp_fold_results["pr_auc"].mean(),
        "roc_auc_mean": mlp_fold_results["roc_auc"].mean(),
        "recall_mean": mlp_fold_results["recall"].mean(),
        "precision_mean": mlp_fold_results["precision"].mean(),
        "f1_mean": mlp_fold_results["f1"].mean(),
        "fit_time_mean_s": mlp_fold_results["fit_time_s"].mean(),
        "score_time_mean_s": mlp_fold_results["score_time_s"].mean(),
    }
])

#display(mlp_fold_results.round(4))
display(mlp_cv_summary.round(4))



,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP_manual,0.6597,0.852,0.8012,0.5287,0.6362,0.6832,0.0036


### ConclusÃƒÂ£o

Na tabela de baselines, a `LogisticRegression` apresentou o melhor desempenho geral, com `PR-AUC = 0.6782` e `ROC-AUC = 0.8576`, ficando levemente acima da `MLP` (`PR-AUC = 0.6728` e `ROC-AUC = 0.8551`). Ainda assim, a diferenÃƒÂ§a entre os dois modelos foi pequena, e a `MLP` superou o benchmark de ÃƒÂ¡rvore selecionado nesta rodada, o `XGBoost` (`PR-AUC = 0.6492`). Isso indica que, jÃƒÂ¡ na base original, a MLP se mostrou competitiva em relaÃƒÂ§ÃƒÂ£o ao baseline linear e ao benchmark nÃƒÂ£o linear.

Na validaÃƒÂ§ÃƒÂ£o da implementaÃƒÂ§ÃƒÂ£o, a `MLP` dentro do `Pipeline` manteve desempenho consistente e atÃƒÂ© ligeiramente superior ÃƒÂ  versÃƒÂ£o manual fora do pipeline, que obteve `PR-AUC = 0.6607` e `ROC-AUC = 0.8528`. Com isso, o wrapper foi validado com sucesso para uso no fluxo de experimentaÃƒÂ§ÃƒÂ£o, trazendo a vantagem de encapsular prÃƒÂ©-processamento e validaÃƒÂ§ÃƒÂ£o cruzada dentro da mesma estrutura, com menor risco de leakage e maior facilidade para evoluir o pipeline com feature engineering e seleÃƒÂ§ÃƒÂ£o de features.


### Logging no MLflow

## Feature Engineering

**Objetivo**: 
- Adicionar poder preditivo aos modelos de forma controlada

### Round 1 - FE orientada a hipÃƒÂ³tese

- adicionando features a partir de hipÃƒÂ³teses construÃƒÂ­das a partir da EDA;

In [12]:
round1_fe_params = dict(
    drop_churn_score=True,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round1_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [13]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÃƒâ€°DIOS DA VALIDAÃƒâ€¡ÃƒÆ’O CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÃƒâ€°DIOS DA VALIDAÃƒâ€¡ÃƒÆ’O CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6893,0.8625,0.8234,0.5382,0.6508,0.0568,0.0209
1,MLP,0.6860,0.8606,0.8203,0.5258,0.6407,0.6905,0.0247
2,XGBoost,0.6465,0.8418,0.6636,0.5752,0.6160,0.0684,0.0278


### Round 2 - Adicionando `Churn Score`

- Verificando o ganho com stacking de outro modelo;

In [14]:
round2_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(
            **round2_fe_params
        )),
        ("geo", GeoTransformer(
            strategy="drop",
        )),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

In [15]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{metric}_mean": cv_res[f"test_{metric}"].mean() for metric in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n=== RESULTADOS MÃƒâ€°DIOS DA VALIDAÃƒâ€¡ÃƒÆ’O CRUZADA ===")
display(results_cv.round(4))

=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: XGBoost ===
=== AVALIANDO MODELO: MLP ===

=== RESULTADOS MÃƒâ€°DIOS DA VALIDAÃƒâ€¡ÃƒÆ’O CRUZADA ===


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,XGBoost,0.9512,0.9802,0.8738,0.8404,0.8566,0.0675,0.0273
1,LogisticRegression,0.9393,0.9759,0.9297,0.7827,0.8497,0.0428,0.0204
2,MLP,0.9368,0.9749,0.9350,0.7521,0.8321,1.0557,0.0228


### Round 3 - Testando encoding para City

Definindo as mesmas features para todos as estratÃƒÂ©gias de encoding

In [26]:
round3_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)


ConfiguraÃ§Ã£o das estratÃ©gias por modelo

In [27]:
assert "City" in X_train_val.columns, "City precisa estar presente em X_train_val para o Round 3."

round3_metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]

round3_strategy_specs = {
    "LogisticRegression": [
        ("frequency", "LogisticRegression_Frequency"),
        ("target", "LogisticRegression_Target"),
        ("geo_cluster", "LogisticRegression_Geo_Cluster"),
        ("zip_region", "LogisticRegression_ZIP"),
        ("risk_band", "LogisticRegression_Risk_Band"),
    ],
    "XGBoost": [
        ("frequency", "XGBoost_Frequency"),
        ("target", "XGBoost_Target"),
        ("geo_cluster", "XGBoost_Geo_Cluster"),
        ("zip_region", "XGBoost_ZIP"),
        ("risk_band", "XGBoost_Risk_Band"),
    ],
    "MLP": [
        ("frequency", "MLP_Frequency"),
        ("target", "MLP_Target"),
        ("geo_cluster", "MLP_Geo_Cluster"),
        ("zip_region", "MLP_ZIP"),
        ("risk_band", "MLP_Risk_Band"),
        ("city_embedding", "MLP_CityEmbedding"),
    ],
}

round3_logistic_params = {
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": 42,
}

round3_xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "scale_pos_weight": (y_train_val == 0).sum() / (y_train_val == 1).sum(),
    "random_state": 42,
    "n_jobs": -1,
}

round3_mlp_params = {
    "hidden_dim": 64,
    "batch_size": 64,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "max_epochs": 80,
    "patience": 8,
    "val_size": 0.15,
    "threshold": 0.5,
    "random_state": 42,
    "verbose": False,
}

round3_embedding_params = {
    "city_column": "City",
    "geo_drop_columns": ("Zip Code", "Latitude", "Longitude", "Lat Long"),
    "embedding_dim": None,
}

In [28]:
round3_results_by_model = {}
round3_fold_results_by_model = {}

for model_name, strategy_specs in round3_strategy_specs.items():
    print(f"=== ROUND 3: {model_name} ===")

    model_results, model_fold_results = evaluate_round3_model_strategies(
        model_name,
        strategy_specs,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        preprocessor=preprocessor,
        fe_params=round3_fe_params,
        y_reference=y_train_val,
        metrics=round3_metrics,
        target_smoothing=20.0,
        logistic_params=round3_logistic_params,
        xgb_params=round3_xgb_params,
        mlp_params=round3_mlp_params,
        embedding_params=round3_embedding_params,
    )

    round3_results_by_model[model_name] = model_results
    round3_fold_results_by_model[model_name] = model_fold_results

=== ROUND 3: LogisticRegression ===
=== ROUND 3: XGBoost ===
=== ROUND 3: MLP ===


Resultados por modelo: LogisticRegression

In [29]:
display(round3_results_by_model["LogisticRegression"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,LogisticRegression_ZIP,0.9394,0.9759,0.9312,0.7835,0.8508,0.0614,0.0212,LogisticRegression,zip_region
1,LogisticRegression_Geo_Cluster,0.9390,0.9757,0.9266,0.7842,0.8493,0.0772,0.0261,LogisticRegression,geo_cluster
2,LogisticRegression_Frequency,0.9390,0.9758,0.9266,0.7807,0.8472,0.0446,0.0207,LogisticRegression,frequency
3,LogisticRegression_Target,0.9221,0.9689,0.8777,0.7792,0.8253,0.0798,0.0234,LogisticRegression,target
4,LogisticRegression_Risk_Band,0.9147,0.9644,0.8716,0.7712,0.8181,0.0436,0.0215,LogisticRegression,risk_band


Resultados por modelo: XGBoost

In [30]:
display(round3_results_by_model["XGBoost"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,XGBoost_Geo_Cluster,0.9527,0.9809,0.8838,0.8468,0.8648,0.0994,0.0369,XGBoost,geo_cluster
1,XGBoost_ZIP,0.9515,0.9808,0.8823,0.8493,0.8654,0.0945,0.0309,XGBoost,zip_region
2,XGBoost_Frequency,0.9493,0.9799,0.8776,0.8478,0.8623,0.0731,0.0293,XGBoost,frequency
3,XGBoost_Risk_Band,0.9356,0.9731,0.8494,0.8374,0.8432,0.0734,0.0295,XGBoost,risk_band
4,XGBoost_Target,0.9330,0.9719,0.8234,0.8559,0.8392,0.0766,0.0293,XGBoost,target


Resultados por modelo: MLP

In [31]:
display(round3_results_by_model["MLP"].round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,MLP_Frequency,0.9372,0.9751,0.9182,0.7761,0.8409,1.2305,0.0260,MLP,frequency
1,MLP_ZIP,0.9362,0.9746,0.9373,0.7610,0.8395,1.1148,0.0257,MLP,zip_region
2,MLP_Geo_Cluster,0.9358,0.9743,0.9212,0.7718,0.8394,1.2793,0.0340,MLP,geo_cluster
3,MLP_CityEmbedding,0.9229,0.9681,0.9380,0.7046,0.8036,1.0582,0.0242,MLP,city_embedding
4,MLP_Target,0.9176,0.9674,0.8990,0.7678,0.8273,1.3768,0.0260,MLP,target
5,MLP_Risk_Band,0.9077,0.9613,0.8792,0.7505,0.8092,1.0749,0.0240,MLP,risk_band


Resultado consolidado do Round 3

In [32]:
round3_results_all = (
    pd.concat(round3_results_by_model.values(), ignore_index=True)
    .sort_values(["base_model", "pr_auc_mean"], ascending=[True, False])
    .reset_index(drop=True)
)

display(round3_results_all.round(4))

,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s,base_model,strategy
0,LogisticRegression_ZIP,0.9394,0.9759,0.9312,0.7835,0.8508,0.0614,0.0212,LogisticRegression,zip_region
1,LogisticRegression_Geo_Cluster,0.9390,0.9757,0.9266,0.7842,0.8493,0.0772,0.0261,LogisticRegression,geo_cluster
2,LogisticRegression_Frequency,0.9390,0.9758,0.9266,0.7807,0.8472,0.0446,0.0207,LogisticRegression,frequency
3,LogisticRegression_Target,0.9221,0.9689,0.8777,0.7792,0.8253,0.0798,0.0234,LogisticRegression,target
4,LogisticRegression_Risk_Band,0.9147,0.9644,0.8716,0.7712,0.8181,0.0436,0.0215,LogisticRegression,risk_band
5,MLP_Frequency,0.9372,0.9751,0.9182,0.7761,0.8409,1.2305,0.0260,MLP,frequency
6,MLP_ZIP,0.9362,0.9746,0.9373,0.7610,0.8395,1.1148,0.0257,MLP,zip_region
7,MLP_Geo_Cluster,0.9358,0.9743,0.9212,0.7718,0.8394,1.2793,0.0340,MLP,geo_cluster
8,MLP_CityEmbedding,0.9229,0.9681,0.9380,0.7046,0.8036,1.0582,0.0242,MLP,city_embedding
9,MLP_Target,0.9176,0.9674,0.8990,0.7678,0.8273,1.3768,0.0260,MLP,target


### ConclusÃ£o

As variÃ¡veis geogrÃ¡ficas nÃ£o demonstraram ganho robusto o suficiente para justificar sua incorporaÃ§Ã£o no pipeline final. Embora algumas estratÃ©gias, como zip_region na RegressÃ£o LogÃ­stica e geo_cluster no XGBoost, tenham produzido pequenas melhoras marginais, os ganhos foram muito discretos e inconsistentes entre os modelos. Na MLP, inclusive, abordagens mais sofisticadas como CityEmbedding elevaram o recall, mas com perda relevante de precisÃ£o. Considerando o aumento de complexidade, o risco de instabilidade e o baixo retorno incremental observado, a decisÃ£o Ã© nÃ£o incluir as variÃ¡veis geogrÃ¡ficas na versÃ£o final do conjunto de features.

### Round 4 - Feature Selection

In [33]:
assert "City" in X_train_val.columns, "City precisa estar presente em X_train_val para esta rodada."
assert "Churn Score" in X_train_val.columns, "Churn Score precisa estar presente em X_train_val para esta rodada."
assert "CLTV" not in X_train_val.columns, "CLTV deve permanecer fora de X_train_val como metadata."

selector_label_map = {
    f_classif: "f_classif",
    mutual_info_classif: "mutual_info_classif",
}

round4_fe_params = dict(
    drop_churn_score=False,
    add_engagement_score=True,
    add_tenure_group=True,
    add_tenure_log=True,
    add_contract_ordinal=True,
    add_family_stability=True,
    add_fiber_no_support=True,
    add_support_gap_count=True,
    add_payment_automatic_flag=True,
    add_electronic_check_flag=True,
    add_paperless_echeck_flag=True,
    add_price_pressure_ratio=True,
)

round4_fe = FeatureEngineerTransformer(**round4_fe_params)
round4_geo = GeoTransformer(strategy="drop")

X_train_val_round4 = round4_fe.fit_transform(X_train_val, y_train_val)
X_train_val_round4 = round4_geo.fit_transform(X_train_val_round4, y_train_val)

processed_feature_names = get_processed_feature_names(
    preprocessor,
    X_train_val_round4,
    y_train_val,
)

k_grid = build_k_grid(
    n_features_processed=len(processed_feature_names),
    min_k=10,
    include_all=True,
)

models = {
    "LogisticRegression": Pipeline([
        ("fe", FeatureEngineerTransformer(**round4_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("selector", SelectKBest()),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),
    "XGBoost": Pipeline([
        ("fe", FeatureEngineerTransformer(**round4_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("selector", SelectKBest()),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    "MLP": Pipeline([
        ("fe", FeatureEngineerTransformer(**round4_fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("selector", SelectKBest()),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", MLPClassifierWrapper(
            hidden_dim=64,
            batch_size=64,
            lr=1e-3,
            weight_decay=1e-5,
            max_epochs=80,
            patience=8,
            val_size=0.15,
            threshold=0.5,
            random_state=42,
            verbose=False,
        )),
    ]),
}

selector_param_grid = {
    "selector__score_func": [f_classif, mutual_info_classif],
    "selector__k": k_grid,
}

rows = []
feature_selection_searches = {}
selected_feature_logs = {}

for model_name, estimator in models.items():
    print(f"Otimizando seletor de features para {model_name}...")

    search = GridSearchCV(
        estimator=estimator,
        param_grid=selector_param_grid,
        cv=cv,
        scoring=scoring,
        refit="recall",
        return_train_score=False,
        n_jobs=1,
    )
    search.fit(X_train_val, y_train_val)

    feature_selection_searches[model_name] = search
    assert hasattr(search, "best_params_")

    selected_features = extract_selected_feature_names(
        search.best_estimator_,
        processed_feature_names,
        selector_step="selector",
    )
    selected_feature_logs[model_name] = selected_features

    best_k = search.best_params_["selector__k"]
    if best_k == "all":
        assert len(selected_features) == len(processed_feature_names)
    else:
        assert len(selected_features) == best_k

    rows.append(
        summarize_grid_search_results(
            search,
            model_name,
            selector_label_map=selector_label_map,
        )
    )

    print(
        format_selected_features_log(
            model_name,
            search.best_params_,
            selected_features,
        )
    )
    print()

results_fs = (
    pd.DataFrame(rows)
    .sort_values("recall_mean", ascending=False)
    .reset_index(drop=True)
)

assert len(results_fs) == 3, "A rodada de feature selection deve retornar exatamente 3 modelos."

print("=== RESULTADOS FEATURE SELECTION ===")
display(results_fs.round(4))

Otimizando seletor de features para LogisticRegression...
=== FEATURES SELECIONADAS: LogisticRegression ===
Seletor: mutual_info_classif
k vencedor: 31
Quantidade final: 31
Features selecionadas:
- cat__Dependents_No
- cat__Dependents_Yes
- cat__Internet Service_Fiber optic
- cat__Internet Service_No
- cat__Online Security_No
- cat__Online Security_No internet service
- cat__Online Backup_No
- cat__Online Backup_No internet service
- cat__Device Protection_No
- cat__Device Protection_No internet service
- cat__Tech Support_No
- cat__Tech Support_No internet service
- cat__Streaming TV_No internet service
- cat__Contract_Month-to-month
- cat__Contract_One year
- cat__Contract_Two year
- cat__Paperless Billing_Yes
- cat__Payment Method_Electronic check
- num__Tenure Months
- num__Monthly Charges
- num__Total Charges
- num__Churn Score
- num__Tenure_Group_Ordinal
- num__Tenure_Log
- num__Contract_Ordinal
- num__Fiber_No_Support
- num__Support_Gap_Count
- num__Payment_Automatic_Flag
- num_

,model,selector,k,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,MLP,mutual_info_classif,25,0.9361,0.9749,0.9449,0.7645,0.8446,2.0497,0.0241
1,LogisticRegression,mutual_info_classif,31,0.9395,0.9763,0.9358,0.7848,0.8534,0.4788,0.0219
2,XGBoost,f_classif,18,0.9545,0.9818,0.8991,0.8410,0.8687,0.0609,0.0272


L1-Based Selection - RegressÃ£o LogÃ­stica

**Regras**
- Limitar a RegularizaÃ§Ã£o para nÃ£o passar de 10 features (mÃ­nimo exigido pelo projeto) -> EarlyStopping

In [37]:
import logging
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from src.features.feature_engineer_transformer import FeatureEngineerTransformer
from src.features.geo_transformer import GeoTransformer


# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------
logger = logging.getLogger("round4_l1_logreg")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def build_l1_logreg_pipeline(C, fe_params):
    return Pipeline([
        ("fe", FeatureEngineerTransformer(**fe_params)),
        ("geo", GeoTransformer(strategy="drop")),
        ("prep", preprocessor),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", LogisticRegression(
            solver="saga",
            l1_ratio=1.0,
            C=C,
            max_iter=5000,
            class_weight="balanced",
            random_state=42,
        )),
    ])


def get_nonzero_feature_names(fitted_pipeline):
    feature_names = fitted_pipeline.named_steps["prep"].get_feature_names_out()
    coefs = fitted_pipeline.named_steps["model"].coef_.ravel()
    nonzero_mask = coefs != 0
    selected_features = np.asarray(feature_names)[nonzero_mask].tolist()
    return selected_features, coefs


# ------------------------------------------------------------
# Grid de C
# Mais à esquerda = regularização mais fraca
# Mais à direita = regularização mais forte
# A busca para quando < 10 features restarem
# ------------------------------------------------------------
c_grid = [
    10.0, 7.5, 5.0, 3.0, 2.0, 1.0,
    0.75, 0.5, 0.3, 0.2, 0.1,
    0.075, 0.05, 0.04, 0.03, 0.02, 0.01,
    0.0075, 0.005, 0.004, 0.003, 0.002, 0.001,
]

min_features_threshold = 10
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]

rows = []
l1_search_logs = {}
l1_search_fold_results = {}

for C in c_grid:
    logger.info("Iniciando avaliação com L1 LogisticRegression | C=%.5f", C)

    estimator = build_l1_logreg_pipeline(
        C=C,
        fe_params=round4_fe_params,
    )

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
        return_estimator=True,
    )

    fold_df = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    l1_search_fold_results[C] = fold_df

    # Refit no conjunto completo para extrair as features finais selecionadas
    fitted_full = clone(estimator).fit(X_train_val, y_train_val)
    selected_features, coefs = get_nonzero_feature_names(fitted_full)
    n_selected = len(selected_features)

    row = {
        "model": "LogisticRegression_L1",
        "C": C,
        "n_selected_features": n_selected,
        "pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "recall_mean": cv_res["test_recall"].mean(),
        "precision_mean": cv_res["test_precision"].mean(),
        "f1_mean": cv_res["test_f1"].mean(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    }
    rows.append(row)

    l1_search_logs[C] = {
        "selected_features": selected_features,
        "coefficients": coefs,
        "metrics": row,
    }

    logger.info(
        (
            "Resultado | C=%.5f | selected=%d | "
            "PR-AUC=%.4f | ROC-AUC=%.4f | Recall=%.4f | Precision=%.4f | F1=%.4f"
        ),
        C,
        n_selected,
        row["pr_auc_mean"],
        row["roc_auc_mean"],
        row["recall_mean"],
        row["precision_mean"],
        row["f1_mean"],
    )

    logger.info("Features selecionadas (%d): %s", n_selected, selected_features)

    if n_selected < min_features_threshold:
        logger.info(
            (
                "Early stopping acionado: número de features selecionadas "
                "(%d) ficou abaixo do limite de %d."
            ),
            n_selected,
            min_features_threshold,
        )
        break

results_l1_fs = (
    pd.DataFrame(rows)
    .sort_values("recall_mean", ascending=False)
    .reset_index(drop=True)
)

display(results_l1_fs.round(4))


20:55:15 | INFO | Iniciando avaliação com L1 LogisticRegression | C=10.00000
20:55:23 | INFO | Resultado | C=10.00000 | selected=47 | PR-AUC=0.9388 | ROC-AUC=0.9758 | Recall=0.9289 | Precision=0.7821 | F1=0.8490
20:55:23 | INFO | Features selecionadas (47): ['cat__Gender_Female', 'cat__Senior Citizen_No', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__Phone Service_No', 'cat__Phone Service_Yes', 'cat__Multiple Lines_No', 'cat__Multiple Lines_No phone service', 'cat__Internet Service_DSL', 'cat__Internet Service_Fiber optic', 'cat__Internet Service_No', 'cat__Online Security_No internet service', 'cat__Online Security_Yes', 'cat__Online Backup_No internet service', 'cat__Online Backup_Yes', 'cat__Device Protection_No', 'cat__Device Protection_No internet service', 'cat__Device Protection_Yes', 'cat__Tech Support_No', 'cat__Tech Support_No internet service', 'cat__Tech Support_Yes', 'cat__Streaming TV_No', 'cat__Streaming TV_No internet service', 'cat__Streaming TV_Yes', 'cat__Streamin

,model,C,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression_L1,0.0030,6,0.9216,0.9670,0.9526,0.6787,0.7926,0.0678,0.0197
1,LogisticRegression_L1,0.0040,12,0.9285,0.9705,0.9488,0.7075,0.8105,0.0807,0.0202
2,LogisticRegression_L1,0.0050,11,0.9315,0.9721,0.9465,0.7291,0.8236,0.0981,0.0196
3,LogisticRegression_L1,0.0075,13,0.9360,0.9744,0.9442,0.7446,0.8324,0.1531,0.0203
4,LogisticRegression_L1,0.0100,13,0.9378,0.9752,0.9419,0.7574,0.8395,0.1888,0.0205
5,LogisticRegression_L1,0.0200,17,0.9392,0.9760,0.9365,0.7667,0.8430,0.2574,0.0204
6,LogisticRegression_L1,0.0300,24,0.9396,0.9762,0.9358,0.7690,0.8440,0.3338,0.0213
7,LogisticRegression_L1,0.0400,26,0.9399,0.9762,0.9350,0.7717,0.8454,0.3697,0.0211
8,LogisticRegression_L1,0.0500,26,0.9399,0.9762,0.9335,0.7730,0.8455,0.4095,0.0217
9,LogisticRegression_L1,0.0750,28,0.9400,0.9762,0.9327,0.7782,0.8483,0.5005,0.0207


- testar um grid mais específico entre 0.005 e 0.003

In [38]:
c_grid = [round(c, 6) for c in np.linspace(0.005, 0.003, 21)]

min_features_threshold = 10
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]

rows = []
l1_search_logs = {}
l1_search_fold_results = {}

for C in c_grid:
    logger.info("Iniciando avaliação com L1 LogisticRegression | C=%.5f", C)

    estimator = build_l1_logreg_pipeline(
        C=C,
        fe_params=round4_fe_params,
    )

    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
        return_estimator=True,
    )

    fold_df = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{metric: cv_res[f"test_{metric}"] for metric in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    l1_search_fold_results[C] = fold_df

    # Refit no conjunto completo para extrair as features finais selecionadas
    fitted_full = clone(estimator).fit(X_train_val, y_train_val)
    selected_features, coefs = get_nonzero_feature_names(fitted_full)
    n_selected = len(selected_features)

    row = {
        "model": "LogisticRegression_L1",
        "C": C,
        "n_selected_features": n_selected,
        "pr_auc_mean": cv_res["test_pr_auc"].mean(),
        "roc_auc_mean": cv_res["test_roc_auc"].mean(),
        "recall_mean": cv_res["test_recall"].mean(),
        "precision_mean": cv_res["test_precision"].mean(),
        "f1_mean": cv_res["test_f1"].mean(),
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    }
    rows.append(row)

    l1_search_logs[C] = {
        "selected_features": selected_features,
        "coefficients": coefs,
        "metrics": row,
    }

    logger.info(
        (
            "Resultado | C=%.5f | selected=%d | "
            "PR-AUC=%.4f | ROC-AUC=%.4f | Recall=%.4f | Precision=%.4f | F1=%.4f"
        ),
        C,
        n_selected,
        row["pr_auc_mean"],
        row["roc_auc_mean"],
        row["recall_mean"],
        row["precision_mean"],
        row["f1_mean"],
    )

    logger.info("Features selecionadas (%d): %s", n_selected, selected_features)

    if n_selected < min_features_threshold:
        logger.info(
            (
                "Early stopping acionado: número de features selecionadas "
                "(%d) ficou abaixo do limite de %d."
            ),
            n_selected,
            min_features_threshold,
        )
        break

results_l1_fs = (
    pd.DataFrame(rows)
    .sort_values("recall_mean", ascending=False)
    .reset_index(drop=True)
)

display(results_l1_fs.round(4))

21:03:21 | INFO | Iniciando avaliação com L1 LogisticRegression | C=0.00500
21:03:22 | INFO | Resultado | C=0.00500 | selected=11 | PR-AUC=0.9315 | ROC-AUC=0.9721 | Recall=0.9465 | Precision=0.7291 | F1=0.8236
21:03:22 | INFO | Features selecionadas (11): ['cat__Dependents_Yes', 'cat__Internet Service_Fiber optic', 'cat__Online Security_No', 'cat__Tech Support_No', 'cat__Contract_Month-to-month', 'cat__Payment Method_Electronic check', 'num__Churn Score', 'num__Contract_Ordinal', 'num__Electronic_Check_Flag', 'num__Paperless_ECheck_Flag', 'num__Price_Pressure_Ratio']
21:03:22 | INFO | Iniciando avaliação com L1 LogisticRegression | C=0.00490
21:03:22 | INFO | Resultado | C=0.00490 | selected=12 | PR-AUC=0.9312 | ROC-AUC=0.9719 | Recall=0.9465 | Precision=0.7275 | F1=0.8225
21:03:22 | INFO | Features selecionadas (12): ['cat__Dependents_No', 'cat__Dependents_Yes', 'cat__Internet Service_Fiber optic', 'cat__Online Security_No', 'cat__Tech Support_No', 'cat__Contract_Month-to-month', 'cat

,model,C,n_selected_features,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression_L1,0.0033,9,0.9245,0.9684,0.9534,0.6906,0.8009,0.0692,0.0190
1,LogisticRegression_L1,0.0035,10,0.9259,0.9692,0.9526,0.6930,0.8023,0.0730,0.0189
2,LogisticRegression_L1,0.0034,10,0.9253,0.9688,0.9526,0.6931,0.8023,0.0722,0.0193
3,LogisticRegression_L1,0.0036,11,0.9265,0.9695,0.9526,0.6977,0.8054,0.0740,0.0191
4,LogisticRegression_L1,0.0038,11,0.9276,0.9701,0.9511,0.7041,0.8091,0.0782,0.0198
5,LogisticRegression_L1,0.0037,11,0.9270,0.9697,0.9511,0.7013,0.8073,0.0758,0.0192
6,LogisticRegression_L1,0.0043,13,0.9297,0.9711,0.9503,0.7168,0.8172,0.0862,0.0197
7,LogisticRegression_L1,0.0042,12,0.9293,0.9709,0.9495,0.7138,0.8149,0.0846,0.0200
8,LogisticRegression_L1,0.0039,13,0.9281,0.9703,0.9495,0.7065,0.8102,0.0804,0.0198
9,LogisticRegression_L1,0.0041,13,0.9289,0.9707,0.9488,0.7104,0.8124,0.0827,0.0199


### ConclusÃƒÂ£o

## Fine Tunning de HiperparÃƒÂ¢metros

**Objetivo:**
- Obter a versÃƒÂ£o otimizada da MLP e do XGBoost (benchmark);
- Definir EarlyStopping para as otimizaÃƒÂ§ÃƒÂµes e rodÃƒÂ¡-las durante 1 hora;
- Avaliar os modelos e comparar com o Baseline (RegressÃƒÂ£o LogÃƒÂ­stica);
- Fazer o Log dos experimentos no MLFlow

### Grid: MLP

### Grid: XGBoost

### AvaliaÃƒÂ§ÃƒÂ£o

### ConclusÃƒÂ£o

# Persistindo o Melhor Modelo